In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# # Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# # Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# # kagglehub.dataset_download('<owner>/<dataset-slug>')

**Import modules**

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings("ignore")

In [ ]:
import os
import gc
import time
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from tqdm import tqdm
import time





import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# Kaggle Secrets for WandB
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    WANDB_TOKEN = user_secrets.get_secret("WANDB_TOKEN")
    wandb.login(key=WANDB_TOKEN)
except:
    print("WandB login running")

**Data loading**

In [ ]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')



In [ ]:
CONFIG = {
    "max_len": 128,
    "vocab_size": 25000,
    "embed_dim": 128,
    "num_filters": 100,
    "filter_sizes": [3, 4, 5],
    "dropout": 0.3,
    "batch_size": 32,
    "epochs": 15,
    "lr": 1e-3,
    "seed": 42,
    "project": "DL-22f1000828-notebook-t22026",
    "entity": "22f1000828-t1-2026-indian-institute-of-technology-madras",
    "device": torch.device('cuda' if torch.cuda.is_available() else 'cpu')
}

LABEL2IDX = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
IDX2LABEL = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True

set_seed(CONFIG['seed'])
print(f"Using Device: {CONFIG['device']}")

**EDA Part**

In [ ]:
print("Train shape:", train.shape)
print("Test shape:", test.shape)

print("Missing values in Train:")
print(train.isnull().sum())

print("Missing values in Test:")
print(test.isnull().sum())

# Check for duplicate questions
print("Duplicate prompts in Train:", train.duplicated(subset=['prompt']).sum())

**Check answers distribution**

In [ ]:
# Are the correct answers evenly spread across A, B, C, D, E?
#Checks if the correct answers are evenly spread or if a specific choice is favored.

plt.figure(figsize=(7, 4))
sns.countplot(data=train, x='answer', palette='Set2')
plt.title('Distribution of Correct Answers')
plt.xlabel('Option Choice')
plt.ylabel('Count')
plt.show()

# Print percentages
print("Answer Distribution (%):")
print(train['answer'].value_counts(normalize=True) * 100)

**Option length bias**

In [ ]:
# Question: Are correct answers usually longer or more detailed than wrong options?
option_cols = ['A', 'B', 'C', 'D', 'E']

# Count words in prompt and each option
train['prompt_len'] = train['prompt'].apply(lambda x: len(str(x).split()))

for col in option_cols:
    train[f'len_{col}'] = train[col].apply(lambda x: len(str(x).split()))

# Compare correct option length vs average wrong option length
def calculate_lengths(row):
    correct_ans = row['answer']
    correct_len = row[f'len_{correct_ans}']
    
    wrong_cols = [c for c in option_cols if c != correct_ans]
    avg_wrong_len = np.mean([row[f'len_{c}'] for c in wrong_cols])
    
    return pd.Series([correct_len, avg_wrong_len])

train[['correct_len', 'avg_wrong_len']] = train.apply(calculate_lengths, axis=1)

print("Average word count:")
print(f"Correct Options   : {train['correct_len'].mean():.2f} words")
print(f"Wrong Options (avg): {train['avg_wrong_len'].mean():.2f} words")

# Plot length distributions
plt.figure(figsize=(8, 4))
sns.kdeplot(train['correct_len'], label='Correct Option', fill=True)
sns.kdeplot(train['avg_wrong_len'], label='Wrong Options (Avg)', fill=True)
plt.title('Word Count: Correct Option vs Wrong Options')
plt.xlabel('Word Count')
plt.legend()
plt.show()

**Trick questions (Negation Words)**

In [ ]:
# Check for words like NOT, EXCEPT, INCORRECT in the prompt
negation_words = ['NOT', 'EXCEPT', 'INCORRECT', 'FALSE', 'LEAST']
pattern = r'\b(' + '|'.join(negation_words) + r')\b'

train['has_negation'] = train['prompt'].apply(
    lambda x: bool(re.search(pattern, str(x), re.IGNORECASE))
)

neg_count = train['has_negation'].sum()
print(f"\nQuestions with negation words (NOT/EXCEPT): {neg_count} ({neg_count / len(train) * 100:.1f}%)")

# Compare answer choices for regular vs negation questions
plt.figure(figsize=(8, 4))
sns.countplot(data=train, x='answer', hue='has_negation')
plt.title('Answer Choices: Regular vs Negation Questions')
plt.xlabel('Answer Choice')
plt.legend(title='Has Negation')
plt.show()

**Prompt and options overlap**

In [ ]:
# Do correct options share more words with the question prompt
#Tests if simple keyword matching helps choose the right answer or 
#if distractors deliberately reuse words from the question to trick the model

def get_word_overlap(prompt, option):
    prompt_words = set(str(prompt).lower().split())
    option_words = set(str(option).lower().split())
    if not option_words:
        return 0
    # Proportion of option words present in the prompt
    return len(prompt_words.intersection(option_words)) / len(option_words)

# Calculate overlap for each option
for col in option_cols:
    train[f'overlap_{col}'] = train.apply(lambda r: get_word_overlap(r['prompt'], r[col]), axis=1)

# What if we simply guess the option with the highest word overlap?
train['highest_overlap_choice'] = (
    train[[f'overlap_{col}' for col in option_cols]]
    .idxmax(axis=1)
    .str.replace('overlap_', '')
)

simple_overlap_acc = (train['highest_overlap_choice'] == train['answer']).mean()
print(f"Accuracy of choosing option with highest word overlap: {simple_overlap_acc * 100:.2f}%")

**Train & Test comparision**

In [ ]:
#Verifies that train and test prompt lengths follow a similar pattern so the model generalizes well

test['prompt_len'] = test['prompt'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(8, 4))
sns.kdeplot(train['prompt_len'], label='Train Prompt Length', fill=True)
sns.kdeplot(test['prompt_len'], label='Test Prompt Length', fill=True)
plt.title('Prompt Length Comparison (Train vs Test)')
plt.xlabel('Word Count')
plt.legend()
plt.show()

**Tokenizer & Dataset Classes**

In [ ]:
# SIMPLE VOCABULARY & TOKENIZER

class SimpleTokenizer:
    def __init__(self, vocab_size=25000):
        self.vocab_size = vocab_size
        self.word2idx = {'<pad>': 0, '<unk>': 1}
        self.idx2word = {0: '<pad>', 1: '<unk>'}
        
    def tokenize(self, text):
        return re.findall(r'\w+', str(text).lower())
        
    def build_vocab(self, texts):
        counter = Counter()
        for text in texts:
            counter.update(self.tokenize(text))
            
        most_common = counter.most_common(self.vocab_size - 2)
        for idx, (word, _) in enumerate(most_common, start=2):
            self.word2idx[word] = idx
            self.idx2word[idx] = word
            
    def encode(self, text, max_len):
        tokens = self.tokenize(text)
        ids = [self.word2idx.get(tok, 1) for tok in tokens[:max_len]]
        if len(ids) < max_len:
            ids = ids + [0] * (max_len - len(ids))
        return ids


# MCQ DATASET CLASS

class MCQCNNDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, is_train=True):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_train = is_train
        self.options = ['A', 'B', 'C', 'D', 'E']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        
        option_tokens = []
        for opt in self.options:
            combined_text = prompt + " " + str(row[opt])
            tokens = self.tokenizer.encode(combined_text, self.max_len)
            option_tokens.append(tokens)

        item = {
            'input_ids': torch.tensor(option_tokens, dtype=torch.long) # Shape: (5, max_len)
        }

        if self.is_train and 'answer' in row:
            item['labels'] = torch.tensor(LABEL2IDX[row['answer']], dtype=torch.long)

        return item

**Model Architecture**

In [ ]:
class MCQTextCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_filters, filter_sizes, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=k)
            for k in filter_sizes
        ])
        
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(len(filter_sizes) * num_filters, 1)

    def forward(self, x):
        batch_size, num_options, seq_len = x.shape
        x = x.view(-1, seq_len)
        
        embeds = self.embedding(x)
        embeds = embeds.permute(0, 2, 1)
        
        pooled_outputs = []
        for conv in self.convs:
            c = F.relu(conv(embeds))
            p = F.max_pool1d(c, kernel_size=c.shape[2]).squeeze(2)
            pooled_outputs.append(p)
            
        cat = torch.cat(pooled_outputs, dim=1)
        cat = self.dropout(cat)
        
        logits = self.fc(cat)
        logits = logits.view(batch_size, num_options)
        return logits

**Installing Weights & Biases**

In [ ]:
!pip uninstall -y wandb
!pip install --no-cache-dir wandb==0.22.3

In [ ]:
import wandb
print(wandb.__version__)

**Data Preparation & Training**

In [ ]:
#Vocabulary & Dataloaders

full_df = train

print("Building Vocabulary...")
tokenizer = SimpleTokenizer(vocab_size=CONFIG['vocab_size'])
all_texts = full_df['prompt'].tolist() + full_df['A'].tolist() + full_df['B'].tolist() + full_df['C'].tolist() + full_df['D'].tolist() + full_df['E'].tolist()
tokenizer.build_vocab(all_texts)

train_df, val_df = train_test_split(
    full_df, 
    test_size=0.15, 
    random_state=CONFIG['seed'], 
    stratify=full_df['answer']
)

train_dataset = MCQCNNDataset(train_df, tokenizer, CONFIG['max_len'], is_train=True)
val_dataset = MCQCNNDataset(val_df, tokenizer, CONFIG['max_len'], is_train=True)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

#Initialize and train model

model = MCQTextCNN(
    vocab_size=len(tokenizer.word2idx),
    embed_dim=CONFIG['embed_dim'],
    num_filters=CONFIG['num_filters'],
    filter_sizes=CONFIG['filter_sizes'],
    dropout=CONFIG['dropout']
)
model.to(CONFIG['device'])

optimizer = AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

wandb.init(
    project="smart-mcq-solver-cnn",
    name=f"1d-text-cnn-{int(time.time())}",
    config=CONFIG
)

best_val_acc = 0.0
print(f"\nStarting 1D-CNN Training for {CONFIG['epochs']} Epochs...\n")

for epoch in range(CONFIG['epochs']):
    model.train()
    total_train_loss = 0
    train_preds, train_targets = [], []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']} [Train]"):
        input_ids = batch['input_ids'].to(CONFIG['device'])
        labels = batch['labels'].to(CONFIG['device'])

        optimizer.zero_grad()
        logits = model(input_ids)
        loss = criterion(logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_train_loss += loss.item()
        preds = torch.argmax(logits, dim=1).detach().cpu().numpy()
        train_preds.extend(preds)
        train_targets.extend(labels.cpu().numpy())

    avg_train_loss = total_train_loss / len(train_loader)
    train_acc = accuracy_score(train_targets, train_preds)

    #Validation 
    model.eval()
    total_val_loss = 0
    val_preds, val_targets = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{CONFIG['epochs']} [Val]"):
            input_ids = batch['input_ids'].to(CONFIG['device'])
            labels = batch['labels'].to(CONFIG['device'])

            logits = model(input_ids)
            loss = criterion(logits, labels)

            total_val_loss += loss.item()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(labels.cpu().numpy())

    avg_val_loss = total_val_loss / len(val_loader)
    val_acc = accuracy_score(val_targets, val_preds)

    print(f"\nEpoch {epoch+1} Results:")
    print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss  : {avg_val_loss:.4f} | Val Acc  : {val_acc:.4f}\n")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "train_acc": train_acc,
        "val_loss": avg_val_loss,
        "val_acc": val_acc
    })

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_cnn_model.pth")
        print(f"Saved Best CNN Model Checkpoint (Val Acc: {val_acc:.4f})")

print("Training Complete.")
wandb.finish()

**Inference & Submission**

In [ ]:
test_df = test

test_dataset = MCQCNNDataset(test_df, tokenizer, CONFIG['max_len'], is_train=False)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'], shuffle=False)

# Load saved model weights
model = MCQTextCNN(
    vocab_size=len(tokenizer.word2idx),
    embed_dim=CONFIG['embed_dim'],
    num_filters=CONFIG['num_filters'],
    filter_sizes=CONFIG['filter_sizes'],
    dropout=CONFIG['dropout']
)

state_dict = torch.load("best_cnn_model.pth", map_location=CONFIG['device'])
model.load_state_dict(state_dict)
model.to(CONFIG['device'])
model.eval()

all_probs = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Generating Test Predictions"):
        input_ids = batch['input_ids'].to(CONFIG['device'])
        logits = model(input_ids)
        probs = F.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)

all_probs = np.vstack(all_probs)
predicted_indices = np.argmax(all_probs, axis=1)
predicted_letters = [IDX2LABEL[idx] for idx in predicted_indices]

submission = pd.DataFrame({
    'id': test_df['id'] if 'id' in test_df.columns else np.arange(len(test_df)),
    'answer': predicted_letters
})

submission.to_csv("submission.csv", index=False)
print("submission.csv' generated successfully")
print(submission.head())